# Documentation: Petroineos Trading Limited — Power Plants Task
## Summer Internship 2026

## My Approach

## <ins>Initialisation</ins>
- Initialising the database csv file name
- Initialising the variable needed to hold the dataframe ready to be saved into excel
- Initialising the variable which helps distinguish the difference between the datasets
- Initialising the variable which retrieves the most database csv file
- Initialising the dictionary which holds the map to convert the country to the full name

## <ins>Analysing the raw data (analyse_plant_data)</ins>
- Loads the raw csv power plant data using encoding='utf-8-sig' used to handle BOM (Byte Order Mark) character in this case "\ufeffDate" was found in source files
- Standardises the column names and removes white spaces
- Convert country codes to full names using dictionary
- Removes duplicates
- Fills missing data with 0's
- Removes rows with all columns as 0 as it means entire row is empty and will not contribute to further analysis


## <ins>Preparing the new data (load_new_data_from_file)</ins>
- Formatting dates to ensure it is in the correct format
- Adds the updatetime/updated by columns
- Reorders column for consistency of database

## <ins>Saving to the database(save_new_data)</ins>
- Takes new data from "load_new_data_from_file" and creates column called "filefrom" to handle database updates without duplication and appends to current database
- The database gets saved with the latest power plant data

## <ins>Get latest data from database (get_data_from_database)</ins>
- Loads the latest data from database and gets stored in internal memory for aggregation functions


## <ins>Getting the quartley statistics for each site (aggregate_data_to_quarterly)</ins>
- Takes the latests data from the database file and groups the data by Quarter and Sitename
- Calculates statistical metrics for each quarter and corresponding site name
- Transforms data to match example given in pdf- however needs a for loop when converting columns. Could become slow if number of sites reach very high numbers

## <ins>Aggregating the data by country and technology (aggregate_data_to_country)</ins>
- Pivot table summing volume by country and technology


In [1]:
import pandas as pd
import numpy as np

In [14]:
class PowerPlants(object):
    def __init__(self):
        self.database_file = 'database.csv'
        self.newdataframe = None
        self.filename = None
        self.datafromdatabase=None
        self.country_dictionary = {'GB': 'Great Britain','FR':'France'}
        
    def analyse_plant_data(self, file_path: str):
        
        """Clean and validate plant data from CSV files."""
        
        
        #Open excel file
        raw_data = pd.read_csv(file_path, encoding='utf-8-sig')
        
        #Removes .csv from end of file name
        self.filename =file_path[:-4]
        
        #To ensure column names stay consistent "Country" column came in as "Country " while others didn't have the space
        raw_data.columns = ['Date', 'Country', 'Technology', 'SiteName', 'Volume']
        
        #Original number of rows
        original_number_rows = raw_data.shape[0]
        
        #Site Name/s:
        list_of_sitenames = raw_data.iloc[np.where(pd.isna(raw_data['SiteName']) == False)]['SiteName'].unique().tolist()
        string_of_sitenames = ", ".join(list_of_sitenames)
        
        #Technology
        list_of_technology = raw_data.iloc[np.where(pd.isna(raw_data['Technology']) == False)]['Technology'].unique().tolist()
        string_of_technology= ", ".join(list_of_technology)
        
        #Change country format
        raw_data['Country']=raw_data['Country'].replace(self.country_dictionary)
        
        #Country
        list_of_country = raw_data.iloc[np.where(pd.isna(raw_data['Country']) == False)]['Country'].unique().tolist()
        
        string_of_country= ", ".join(list_of_country)
        
        #Drop duplicates
        raw_data.drop_duplicates(inplace=True)
        
        
        
        #Fill any non-numeric (string/NaN) data from Volume column with 0
        raw_data['Volume'] = pd.to_numeric(raw_data['Volume'],errors='coerce').fillna(0)
        
        #Number of missing volume
        missing_volumes = (raw_data['Volume'] == 0).sum()
        
        
        #Fill NaN with 0
        raw_data.fillna(0,inplace=True)
        
        #Drop any row with all zeros because it means all values came in as empty and will be unuseful
        raw_data =raw_data.loc[~(raw_data == 0).all(axis=1)]
        
        
        
        
        #New number of rows
        new_number_rows = raw_data.shape[0]
        
        fstring =  f"""Name: {file_path[:-4]}
       Country: {string_of_country}
       Original Number of rows: { original_number_rows}
       New Number of rows: {new_number_rows}
       Rows Removed: {original_number_rows-new_number_rows}
       Number of missing volume: {missing_volumes}
       Site Name/s: {string_of_sitenames}
       Technology: {string_of_technology}
       Data is now clean
                  """
        print(fstring)
        return raw_data
    
    def load_new_data_from_file(self, input_data: pd.DataFrame):
        
        """Prepare data for saving to database."""
        
        #Retrieve Data
        newdataframe = input_data
        #Change date format to YYYY-MM-DD
        newdataframe['Date'] = pd.to_datetime(newdataframe['Date'], format="%d/%m/%Y").dt.date
        
        #Create updatetime column
        newdataframe['updatetime'] = pd.Timestamp.today()
        #Create updatedby column
        newdataframe['updatedby'] = 'petroineos'
        #Round Volme to 6 decimal places
        newdataframe['Volume'] = newdataframe['Volume'].astype(float).round(6)
        
        #Change order to columns
        newdataframe = newdataframe[['Date','Country','SiteName','Technology','updatedby','updatetime','Volume']]
        #Rename columns order to columns
        newdataframe.columns =['date','country','SiteName','Technology','updatedby','updatetime','Volume']
        #Assign new dataframe
        self.newdataframe=newdataframe
        print('Data is now processed and loaded')
        return newdataframe
        
    
    def save_new_data(self, input_data: pd.DataFrame):
        
        """Save data to database.csv, replacing previous entries from same file."""
        
        
        #I added a 'fromfile' column to track which source file each record came from. 
        #This allows safe re-uploading of the same file and the old version is replaced rather than duplicated, 
        #preventing data integrity issues.
        self.newdataframe['fromfile'] =self.filename
        try:
            #Download the current database
            db = pd.read_csv(self.database_file)
            
            #Remove old entries from same file
            db = db.iloc[np.where(db['fromfile'] != self.filename)]
            
            #Append new data
            combineddataframes = pd.concat([self.newdataframe,db])
        
            combineddataframes.to_csv(self.database_file,index=False)
            print(f'Latest version of {self.filename} added to database')
        except FileNotFoundError:
            self.newdataframe.to_csv(self.database_file,index=False)
            print(f'{self.database_file} created and latest version of {self.filename} added to database')
    
    def get_data_from_database(self):
        
        """Return latest data from database."""
        
        try:
            #Download the current database
            self.datafromdatabase = pd.read_csv(self.database_file)
            #Database columns apart from "fromfile"
            dbcols = self.datafromdatabase.columns[:-1]
            print(f'Latest {self.database_file} loaded')
            return self.datafromdatabase[dbcols]
        except FileNotFoundError:
            print(f'{self.database_file} not found please use save_new_data method or create {self.database_file} manually')
    
    def aggregate_data_to_quarterly(self):
        
        """Return latest quarterly mean, median and std for each plant."""
        
        try:
            #latest data from datebase
            databasedata = self.datafromdatabase[['date','SiteName','Volume']]
            
            #Convert date column to datetime format so we can perform quarterly grouping
            databasedata['date'] = pd.to_datetime( databasedata['date'])
            
            #Create column with quarterly dates
            #Q-DEC means Q1 will be from Jan-Mar, Q2 will be Apr-Jun,Q3 will be from Jul-Sep, Q4 will be from Oct-Dec
            databasedata['Quarters'] =databasedata['date'].dt.to_period('Q-DEC')
            
            #Group by quarters and sitename
            #Unstack switches the columns from being vertically stacked to horizontally stacked by each sitename
            #making it easier to read
            databasedata=databasedata.groupby(['Quarters','SiteName']).agg({'Volume': ['mean','median','std']}).unstack(level='SiteName')
            
            #Swap multiindex column so that sitenames can be aggregated together
            databasedata=databasedata.swaplevel(axis=1)
            
            #Aggregate sitenames together with the corresponding statistics and sorts in alphabetical order
            databasedata.sort_index(axis=1,inplace=True)
            
            #Creates columns which are in the same format as "month_average_min_mix.png"
            databasedata.columns = [f'{site} {stat}' for Volume,site, stat in databasedata.columns]
            return databasedata
        
        except TypeError:
            print('please run the method get_data_from_database to get most recent database')
    
    def aggregate_data_to_country(self):
        
        """Return total volume by country and technology."""
        
        
        try:
            #Get total sum of Volume pivoted by country and technology
            aggregatetable=self.datafromdatabase.pivot_table(values='Volume',index=['country','Technology'],aggfunc='sum')
            return aggregatetable
        except TypeError:
            print('please run the method get_data_from_database to get most recent database')
    
    

In [15]:
pp = PowerPlants()
new_data=pp.load_new_data_from_file(pp.analyse_plant_data("wind_plants.csv"))
pp.save_new_data(new_data)
new_data=pp.load_new_data_from_file(pp.analyse_plant_data("gas_plants.csv"))
pp.save_new_data(new_data)
new_data=pp.load_new_data_from_file(pp.analyse_plant_data("gas_fr_plants.csv"))
pp.save_new_data(new_data)


Name: wind_plants
       Country: Great Britain
       Original Number of rows: 957
       New Number of rows: 957
       Rows Removed: 0
       Number of missing volume: 0
       Site Name/s: Hornsea-1, Hornsea-2
       Technology: Wind 
       Data is now clean
                  
Data is now processed and loaded
Latest version of wind_plants added to database
Name: gas_plants
       Country: Great Britain
       Original Number of rows: 962
       New Number of rows: 962
       Rows Removed: 0
       Number of missing volume: 4
       Site Name/s: Pembroke-1, Pembroke-2
       Technology: Gas
       Data is now clean
                  
Data is now processed and loaded
Latest version of gas_plants added to database
Name: gas_fr_plants
       Country: France
       Original Number of rows: 962
       New Number of rows: 481
       Rows Removed: 481
       Number of missing volume: 7
       Site Name/s: Blenod-5
       Technology: Gas
       Data is now clean
                  
Data is 

In [16]:
pp.get_data_from_database()

Latest database.csv loaded


,date,country,SiteName,Technology,updatedby,updatetime,Volume
0,2024-01-01,France,Blenod-5,Gas,petroineos,2026-06-01 10:10:09.000485,6753.000000
1,2024-01-02,France,Blenod-5,Gas,petroineos,2026-06-01 10:10:09.000485,3896.000000
2,2024-01-03,France,Blenod-5,Gas,petroineos,2026-06-01 10:10:09.000485,3636.000000
3,2024-01-04,France,Blenod-5,Gas,petroineos,2026-06-01 10:10:09.000485,5138.000000
4,2024-01-05,France,Blenod-5,Gas,petroineos,2026-06-01 10:10:09.000485,5265.000000
...,...,...,...,...,...,...,...
2395,2025-04-21,Great Britain,Hornsea-2,Wind,petroineos,2026-06-01 10:10:08.885584,711.231619
2396,2025-04-22,Great Britain,Hornsea-2,Wind,petroineos,2026-06-01 10:10:08.885584,808.534585
2397,2025-04-23,Great Britain,Hornsea-2,Wind,petroineos,2026-06-01 10:10:08.885584,142.450340
2398,2025-04-24,Great Britain,Hornsea-2,Wind,petroineos,2026-06-01 10:10:08.885584,392.082184


In [17]:
pp.aggregate_data_to_quarterly()

C:\Users\simso\AppData\Local\Temp\ipykernel_32580\1228266882.py:153: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  databasedata['date'] = pd.to_datetime( databasedata['date'])
C:\Users\simso\AppData\Local\Temp\ipykernel_32580\1228266882.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  databasedata['Quarters'] =databasedata['date'].dt.to_period('Q-DEC')


,Blenod-5 mean,Blenod-5 median,Blenod-5 std,Hornsea-1 mean,Hornsea-1 median,Hornsea-1 std,Hornsea-2 mean,Hornsea-2 median,Hornsea-2 std,Pembroke-1 mean,Pembroke-1 median,Pembroke-1 std,Pembroke-2 mean,Pembroke-2 median,Pembroke-2 std
Quarters,,,,,,,,,,,,,,,
2024Q1,4948.417582,4912.0,1054.780273,489.115071,491.417052,280.556287,470.166309,442.588117,320.428091,7118.450549,7192.0,1081.092033,6893.472527,6824.0,1188.405826
2024Q2,4976.780220,4987.0,1130.818738,458.989280,466.683337,266.749336,477.136650,425.819859,282.352554,7055.450549,6828.0,1193.288652,7086.824176,7097.0,1156.965404
2024Q3,5031.195652,4941.5,1141.124279,2199.987554,540.043540,13021.876011,482.641581,454.421450,265.213415,7210.804348,7311.5,1134.721075,7041.130435,7068.0,1146.210644
2024Q4,4738.282609,5048.5,1711.083400,478.276781,500.668524,293.671778,506.687955,488.632976,285.389968,7071.673913,7294.5,1154.860232,6863.586957,6766.5,1236.674486
2025Q1,5053.255556,5238.5,1215.552693,506.998769,516.975897,287.224672,530.510781,571.584016,290.506335,6816.377778,6769.0,1100.306184,6920.877778,7361.0,1828.154044
2025Q2,4912.200000,4461.0,1287.277813,452.714035,406.341511,278.009828,551.854866,533.487056,316.884377,7011.880000,7006.0,1008.187124,6965.360000,7071.0,1300.411886


## Observations from Quarterly Analysis

- Hornsea-1 shows an unusually high standard deviation in 2024Q3 
  (13,021 vs ~280 in other quarters) also the mean within 2024Q3 is unusally high (2199 vs ~450-500 in other quarters)
- However the median is within an an acceptable range as the other quarters
- This suggests a significant outlier within the data during that time period. This should be cross-referenced with actual wind speed data during that time period to see if the outlier matches reality or if there is a data issue


In [18]:
pp.aggregate_data_to_country()

Volume
country       Technology              
France        Gas         2.379583e+06
Great Britain Gas         6.741038e+06
              Wind        6.259776e+05